# 损失函数教学文档（学生版）

本笔记面向零基础，按“概念 → 公式 → 例子 → 代码 → 练习”展开。

你将学会：
- 损失函数是什么、为什么需要它
- 回归与分类常用损失
- 交叉熵的直觉解释
- 损失与梯度的关系


## 1. 损失函数是什么

**损失函数就是：衡量模型“错了多少”的数字。**

- 预测越接近真实答案，损失越小
- 预测越离谱，损失越大

训练的目标：**让损失尽可能小**。


## 2. 回归 vs 分类

- **回归**：输出连续数值（比如房价）
- **分类**：输出离散类别（比如 0~9）

不同任务使用不同的损失函数。


## 3. 回归任务常用损失

### 3.1 均方误差（MSE）

$$
\mathrm{MSE} = 
rac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2
$$

特点：
- 大误差会被平方放大
- 对异常值敏感

### 3.2 平均绝对误差（MAE）

$$
\mathrm{MAE} = 
rac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|
$$

特点：
- 对异常值不那么敏感
- 但在 0 点不可导

### 3.3 Huber 损失（折中）

在误差小的时候像 MSE，大的时候像 MAE。


In [ ]:
import numpy as np

# 回归任务：真实值 vs 预测值
y_true = np.array([3.0, 2.5, 4.0])
y_pred = np.array([2.5, 2.7, 3.5])

# 均方误差（MSE）：误差平方后取平均
mse = np.mean((y_true - y_pred) ** 2)

# 平均绝对误差（MAE）：误差绝对值取平均
mae = np.mean(np.abs(y_true - y_pred))

print("MSE:", mse)
print("MAE:", mae)


## 4. 分类任务常用损失

### 4.1 多分类交叉熵（Cross Entropy）

模型输出每一类的概率 $p_i$，真实标签用 one-hot 表示 $t_i$：

$$
\mathrm{CE} = -\sum_i t_i \log(p_i)
$$

因为 one-hot 只有一个 1，上式等价于：

$$
\mathrm{CE} = -\log(p_{\text{true}})
$$

直觉：
- 正确类别的概率越大，损失越小
- 正确类别的概率越小，损失越大


### 4.2 二分类交叉熵（Binary Cross Entropy）

$$
\mathrm{BCE} = -ig( y\log(p) + (1-y)\log(1-p) ig)
$$

- 适合只有两类的任务
- 典型输出层：sigmoid


### 4.3 cross_entropy_error 实现（常用版本）

在代码里通常会写一个 `cross_entropy_error`，它做三件事：

1) **支持批量输入**（shape 为 `(batch, num_classes)`）
2) **支持 one-hot 标签** 或 **整数标签**
3) **避免 log(0)**（加一个很小的常数）

公式（批量版）：

$$
L = -\frac{1}{N}\sum_{i=1}^{N}\log\big(p_{i,\text{true}}\big)
$$


In [ ]:
import numpy as np

# 常用的交叉熵实现（支持 one-hot / 标签索引）

def cross_entropy_error(y, t):
    # y: 预测概率，shape (batch, num_classes) 或 (num_classes,)
    # t: 标签（one-hot 或类别索引）
    if y.ndim == 1:
        y = y.reshape(1, -1)
        t = t.reshape(1, -1)

    # 如果 t 是 one-hot，转成索引
    if t.size == y.size:
        t = np.argmax(t, axis=1)

    batch_size = y.shape[0]
    delta = 1e-7  # 防止 log(0)

    return -np.sum(np.log(y[np.arange(batch_size), t] + delta)) / batch_size


## 4.4 Sigmoid 函数（数学表达式）

Sigmoid 函数定义为：

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

特点：

- 输出范围在 $(0, 1)$
- 常用于二分类输出层或作为激活函数
- 当 $x$ 很大时，输出接近 1；当 $x$ 很小时，输出接近 0


In [1]:
import numpy as np

# softmax 与交叉熵示例

def softmax(x):
    # 先减最大值，避免指数溢出
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)

# 模型输出的“得分”（还不是概率）
logits = np.array([1.2, 0.3, 2.1, -0.7])

# 变成概率
probs = softmax(logits)

# 真实类别为 2（从 0 开始）
true_label = 2

# 交叉熵损失：只看正确类别的概率
loss = -np.log(probs[true_label])

print("probs:", probs)
print("cross-entropy:", loss)


probs: [0.24902002 0.10124398 0.61249042 0.03724558]
cross-entropy: 0.49022198542559564


## 5. 为什么不用准确率当损失

准确率是“对/错”，不可微分。

训练需要梯度（导数），所以用可微的损失函数来指导参数更新。


## 6. 数值稳定性（为什么要减最大值）

softmax 里有指数：

$$
\mathrm{softmax}(x_i)=
rac{e^{x_i}}{\sum_j e^{x_j}}
$$

如果 $x$ 很大，$e^{x}$ 可能溢出。

解决办法：

$$
\mathrm{softmax}(x_i)=
rac{e^{x_i-\max(x)}}{\sum_j e^{x_j-\max(x)}}
$$


In [ ]:
import numpy as np

def softmax_stable(x):
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)


## 7. 损失函数与梯度的关系

损失函数是参数的函数：

$$
L = L(W)
$$

梯度就是损失对参数的导数：

$$

rac{\partial L}{\partial W}
$$

训练时更新参数：

$$
W \leftarrow W - \eta 
rac{\partial L}{\partial W}
$$

直觉：损失是“高度”，梯度是“坡度”。沿着负梯度走，损失下降。


## 7.1 为什么能对权重求导（链式法则的直觉）

你看到的损失函数输入只有 **预测值** 和 **真实值**，那为什么能对权重求导？

关键在于：**预测值是由权重算出来的**。

也就是说，损失函数虽然“表面上”只看预测值，但预测值本身取决于权重 $W$。

用公式写就是：

$$
	ext{预测值 }\hat{y} = f(x; W)
$$

$$
L = L(\hat{y}, y_{true}) = L(f(x; W), y_{true})
$$

所以损失函数其实是 **权重的函数**，可以对 $W$ 求导。

### 一个最简单的例子（线性回归）

假设：

$$
\hat{y} = Wx
$$

$$
L = (\hat{y} - y)^2
$$

因为 $\hat{y}$ 依赖 $W$，所以：

$$

rac{dL}{dW} = 2(\hat{y}-y)\cdot x
$$

直觉理解：
- 预测错得越多，更新幅度越大
- 输入 $x$ 越大，对应权重的影响也更大

### 一句话总结

**损失对权重求导，是通过“预测值依赖权重”这一层关系实现的。**


In [ ]:
# 一个最小可运行的例子：损失对权重的导数

x = 2.0       # 输入
w = 3.0       # 权重

y_true = 5.0  # 真实值

# 预测值：由权重和输入计算得到

y_pred = w * x

# 损失：预测与真实的差异

loss = (y_pred - y_true) ** 2

# 对权重的导数（梯度）
# dL/dW = 2*(y_pred - y_true) * x

grad = 2 * (y_pred - y_true) * x

print("y_pred:", y_pred)
print("loss:", loss)
print("dL/dW:", grad)


## 8. 常见误区

- **softmax 输出才是概率**，logits 不是概率
- **交叉熵需要配合正确的标签表示（one-hot 或索引）**
- **损失变小不代表模型一定泛化好**（可能过拟合）


## 9. 练习题（建议动手）

1) 用 MSE 计算以下样本的损失：
   - 真实值 `[1, 2, 3]`
   - 预测值 `[1.5, 1.7, 2.9]`

2) 对一个三分类问题，预测概率是 `[0.1, 0.7, 0.2]`，真实类别是第 1 类（索引 1）。交叉熵损失是多少？

3) 解释：为什么交叉熵对“把正确类概率压得很小”的情况惩罚很重？


## 10. 小结

- 损失函数衡量“预测错了多少”
- 回归常用 MSE/MAE，分类常用交叉熵
- 训练依赖损失的梯度来更新参数
- 选择合适损失函数，决定模型能否学到正确目标


### 4.1.1 one-hot 转类别索引（argmax）

**用途**：把 one-hot 标签转换成类别索引，便于计算损失或准确率。

- one-hot：`[0, 0, 1, 0]` 代表第 2 类
- 类别索引：`2`

**常见场景**：
- 交叉熵损失需要“正确类别的索引”
- 计算准确率时，需要把预测/标签都变成索引

**用法**：

```python
import numpy as np

one_hot = np.array([[0, 0, 1, 0], [1, 0, 0, 0]])
idx = np.argmax(one_hot, axis=1)
print(idx)  # [2 0]
```
